In [2]:
# =====================================================================
# End-to-End ML Pipeline: Google Colab to AWS S3
# =====================================================================
!pip install boto3
import os
import boto3
import joblib
import pandas as pd
from io import StringIO
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 31.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [boto3]


In [11]:
# ---------------------------------------------------------------------
# STEP 1: AWS Configuration & Authentication
# ---------------------------------------------------------------------
# Replace these with your actual AWS credentials.
# Keep these secret! In production, use AWS IAM roles or environment variables.
AWS_ACCESS_KEY = "YOUR_AWS_ACCESS_KEY"
AWS_SECRET_KEY = "YOUR_AWS_SECRET_KEY"
AWS_REGION = "us-east-1"
BUCKET_NAME = "simple-ml-pipeline-bucket" # Must be globally unique

# Initialize the AWS S3 Client
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

In [6]:
# ---------------------------------------------------------------------
# STEP 2: Data Ingestion (Pulling data from a source)
# ---------------------------------------------------------------------
print("--- Step 2: Ingesting Data ---")
# Using a public health dataset (Wine quality classification)
data_url = "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-red.csv"
df = pd.read_csv(data_url, sep=';')

# Convert target variable to a binary classification task (Good quality vs Rest)
df['is_good_quality'] = (df['quality'] >= 7).astype(int)
df = df.drop(columns=['quality'])

X = df.drop(columns=['is_good_quality'])
y = df['is_good_quality']
print(f"Dataset successfully loaded. Shape: {df.shape}")

--- Step 2: Ingesting Data ---
Dataset successfully loaded. Shape: (1599, 12)


In [7]:
# ---------------------------------------------------------------------
# STEP 3: Data Preprocessing & Splitting
# ---------------------------------------------------------------------
print("\n--- Step 3: Preprocessing Data ---")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data split into train/test sets and normalized.")



--- Step 3: Preprocessing Data ---
Data split into train/test sets and normalized.


In [8]:
# ---------------------------------------------------------------------
# STEP 4: Model Training
# ---------------------------------------------------------------------
print("\n--- Step 4: Training Model ---")
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)
print("Random Forest Classifier trained successfully.")


--- Step 4: Training Model ---
Random Forest Classifier trained successfully.


In [9]:
# ---------------------------------------------------------------------
# STEP 5: Model Evaluation
# ---------------------------------------------------------------------
print("\n--- Step 5: Evaluating Model ---")
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f"Pipeline Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


--- Step 5: Evaluating Model ---
Pipeline Accuracy: 90.00%

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.97      0.94       273
           1       0.73      0.51      0.60        47

    accuracy                           0.90       320
   macro avg       0.82      0.74      0.77       320
weighted avg       0.89      0.90      0.89       320



In [10]:
# ---------------------------------------------------------------------
# STEP 6: Serialize & Upload Artifacts to AWS S3
# ---------------------------------------------------------------------
print("\n--- Step 6: Deploying Artifacts to AWS ---")

# Save model and scaler locally to Colab's ephemeral storage first
model_filename = "wine_rf_model.joblib"
scaler_filename = "data_scaler.joblib"

joblib.dump(model, model_filename)
joblib.dump(scaler, scaler_filename)

try:
    # Upload the trained model to your S3 bucket
    s3_client.upload_file(model_filename, BUCKET_NAME, f"models/{model_filename}")
    # Upload the scaler (required to transform future production data)
    s3_client.upload_file(scaler_filename, BUCKET_NAME, f"scalers/{scaler_filename}")

    print(f"🎉 Success! Pipeline complete. Artifacts saved to S3 bucket: '{BUCKET_NAME}'")

except Exception as e:
    print(f"❌ AWS Upload Failed. Verify your credentials and bucket name. Error: {e}")

# Clean up local temporary files in Colab
if os.path.exists(model_filename): os.remove(model_filename)
if os.path.exists(scaler_filename): os.remove(scaler_filename)


--- Step 6: Deploying Artifacts to AWS ---
❌ AWS Upload Failed. Verify your credentials and bucket name. Error: Failed to upload wine_rf_model.joblib to your-colab-ml-pipeline-bucket/models/wine_rf_model.joblib: An error occurred (InvalidAccessKeyId) when calling the PutObject operation: The AWS Access Key Id you provided does not exist in our records.
